# 39. Rosenbaum-type Γ and the step-wise Λ in M0's actual design

Proposition 5 shows that the inclusion R_Γ ⊆ S_Γⁿ is sharp over **all** designs with n alerts and a cooldown; the manuscript leaves open (Appendix H.3) what the constant is in M0's actual design (800 alerts, month × volatility-decile cells, 240-minute cooldown). The exact value is a maximum over 2^L weightings and lies out of computational reach, so here I compute only the quantities within reach, on Binance spot BTCUSDT one-minute data over the development period (2023-01 to 2026-04):

- Cell packing (section 3): for each cell I record the maximum number of alerts K_c that fit under the cooldown, the actual count ν_c and the utilization ν_c/K_c. Density is neither necessary nor sufficient for a large odds ratio, so I treat these as descriptive statistics.
- Lower bounds on the design constant (section 4): for a few fixed weightings v ∈ [1, Γ]^L, I compute the step odds ratio OR_v(t, H) = odds_v / odds_1 by exact dynamic programming at histories reachable under the null Q (the observed M0 placement and the leading parts of placements drawn from Q). Each value is a lower bound on κ_D(Γ) = sup over v, H, t of max(OR, 1/OR), so with m = max |log OR| / log Γ, κ_D(Γ) ≥ Γ^m. Because the inclusion R_Γ ⊆ S_Λ needs Λ ≥ κ_D(Γ), the Γ that the inclusion can certify is then at most 1.795^(1/m).
- Upper bounds on Γ\* (section 5): I compute exact tail probabilities Q_v(T ≥ 22) under fixed weightings of the same kind. Each is a lower bound on the Rosenbaum-type worst-case p, so a value of at least 0.05 at some grid Γ puts Γ* (α = 0.05) below that Γ. The tilt of notebook 35 (Γ on pre-onset minutes) gave Γ* ≤ 3.17; under the cooldown the tilt need not be the worst case (Proposition 4).
- Design export (section 2): I save `39_design.npz` with minute stamps, cells, pre-onset flags, alert counts per cell and M0 alert positions (no prices).

In section 0 I rebuild M0's scores, alerts and events with the code of section 0 of notebooks 34 and 35, and in section 1 I check the odds-ratio kernel against full enumeration on small designs. I write the outputs (`1_kernel_check.csv`, `39_design.npz`, `3_cell_packing.csv`, `3b_month_packing.csv`, `4_path_odds_ratios.csv`, `5_weighted_tail.csv`, `5b_gamma_star_upper.csv`, `39_SUMMARY.json`) to `design_constant_v1`. The reading rules, which I fix in advance in `39_SPEC.json`, limit the results to lower bounds on κ_D(Γ) and upper bounds on Γ*, because a lower bound on Γ* needs a proof that κ_D(Γ) ≤ Γ^k (which would give Γ* ≥ 1.795^(1/k)). This analysis is exploratory, since I have inspected the development sample repeatedly; I leave the registered confirmatory plan and the Λ* report unchanged.

In [1]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df

# Shared with notebook 16 (identical code): VPIN, causal feature set, GBM settings.
REGIME_WINDOW_D = 30; RV_WINDOW = 1440; VPIN_BUCKETS_PER_DAY = 50; VPIN_WINDOW = 50; VPIN_LOOKBACK_D = 30
GBM_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=1.0,
                  class_weight='balanced', early_stopping=False, random_state=17)
FEATS = ['z_vol', 'tsr', 'z_hs', 'ret_5', 'ret_15', 'ret_60', 'ret_240', 'rv_15', 'rv_60', 'rv_240', 'rv_ratio_15_240',
         'dd_240', 'dd_1440', 'vol_intensity_15', 'sell_ratio_15', 'sell_ratio_60', 'vpin']

@njit(cache=False)
def _vpin_core(vol, buy, bucket, n_win):
    out = np.full(len(vol), np.nan); ring = np.zeros(n_win); k = 0; run = 0.0; cv = 0.0; cb = 0.0
    for i in range(len(vol)):
        v = vol[i]; b = buy[i]
        if not (v == v) or not (b == b) or v < 0:
            v = 0.0; b = 0.0
        cv += v; cb += min(max(b, 0.0), v)
        bs = bucket[i]
        if bs == bs and bs > 0 and cv >= bs:
            imb = abs(2.0 * cb - cv) / cv
            j = k % n_win; run += imb - ring[j]; ring[j] = imb; k += 1; cv = 0.0; cb = 0.0
        if k >= n_win:
            out[i] = run / n_win
    return out


def vpin_series(px):
    vol = px.volume.to_numpy(float); buy = px.taker_buy_vol.to_numpy(float)
    daily = px.volume.groupby(px.index.floor('D')).sum()
    size = (daily.rolling(VPIN_LOOKBACK_D, min_periods=VPIN_LOOKBACK_D).mean().shift(1) / VPIN_BUCKETS_PER_DAY)
    bucket = size.reindex(px.index.floor('D')).to_numpy(float)
    return _vpin_core(vol, buy, bucket, VPIN_WINDOW)


def causal_features(px, sg):
    c = px.close.astype(float); lc = np.log(c); r = lc.diff(); vol = px.volume.astype(float); sell = vol - px.taker_buy_vol.astype(float)
    F = pd.DataFrame(index=px.index)
    F['z_vol'] = sg.z_vol.astype(float); F['tsr'] = sg.tsr.astype(float); F['z_hs'] = sg.z_hs.astype(float)
    for k in (5, 15, 60, 240):
        F[f'ret_{k}'] = lc - lc.shift(k)
    for k in (15, 60, 240):
        F[f'rv_{k}'] = r.rolling(k, min_periods=k // 2).std()
    F['rv_ratio_15_240'] = F.rv_15 / F.rv_240
    F['dd_240'] = 1 - c / c.rolling(240, min_periods=120).max()
    F['dd_1440'] = 1 - c / c.rolling(1440, min_periods=720).max()
    F['vol_intensity_15'] = vol.rolling(15, min_periods=8).sum() / (vol.rolling(1440, min_periods=720).sum() / 96)
    F['sell_ratio_15'] = sell.rolling(15, min_periods=8).sum() / vol.rolling(15, min_periods=8).sum()
    F['sell_ratio_60'] = sell.rolling(60, min_periods=30).sum() / vol.rolling(60, min_periods=30).sum()
    F['vpin'] = vpin_series(px)
    return F.replace([np.inf, -np.inf], np.nan).astype('float32')
OUT = SPOT_BASE / 'design_constant_v1'; OUT.mkdir(parents=True, exist_ok=True)
M16 = SPOT_BASE / 'methods_definitions_v1'
M34 = SPOT_BASE / 'exact_conditional_null_v1'
M35 = SPOT_BASE / 'sequential_sensitivity_v1'
DNAME, DCL = '4pct', 0.04; H_REF = 240
SEED = 20261104
C_MIN = COOLDOWN
GAMMA_OR = (1.5, 2.0)                              # section 4: fixed weightings at these Gamma
GAMMA_TAIL = (2.5, 3.0)                            # section 5: exact weighted tails; a weighting beats the tilt (Gamma* <= 3.17, notebook 35) only if p >= 0.05 below 3.17
N_DRAW = 10                                        # section 4: paths drawn from Q in addition to the observed M0 path
BLOCK_RAND_MIN = 60                                # section 4: random-block weighting, block length in minutes
LAMBDA_STAR = 1.7954692410260404                   # notebook 35, alpha = 0.05 (checked against 3b_lambda_star.csv below)

for _p in [M16 / 'METHODS_FROZEN.json', M16 / 'METHODS_SPEC.json', M16 / f'alarms_M0_VF_logit_{DNAME}.csv', M34 / '2_m0_exact_vs_budget.csv',
           M35 / '3b_lambda_star.csv', M35 / '4_rosenbaum_tilt.csv', M35 / '4b_gamma_star_tilt.csv']:
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 16, 34 and 35 are required: {_p}')
FROZEN16 = json.loads((M16 / 'METHODS_FROZEN.json').read_text())
if sha256_file(M16 / 'METHODS_SPEC.json') != FROZEN16['spec_sha256']:
    raise RuntimeError('GUARD FAIL: notebook-16 spec differs')
REF34 = pd.read_csv(M34 / '2_m0_exact_vs_budget.csv').iloc[0]
STAR35 = pd.read_csv(M35 / '3b_lambda_star.csv'); TILT35 = pd.read_csv(M35 / '4_rosenbaum_tilt.csv'); GSTAR35 = pd.read_csv(M35 / '4b_gamma_star_tilt.csv')
if not np.isclose(float(STAR35.loc[STAR35.alpha == 0.05, 'lambda_star'].iloc[0]), LAMBDA_STAR, rtol=1e-12):
    raise RuntimeError('GUARD FAIL: notebook-35 lambda* differs from the value fixed above')

SPEC = {'title': 'Design-specific relation between a Rosenbaum-type Gamma and the step-wise Lambda in M0\'s development design (exploratory; appendix)',
        'null_model': 'Q_nu of notebook 34 (uniform over alert sets with the observed counts per calendar month x RV decile and spacing >= 240 minutes)',
        'outputs': {'packing': 'per cell: eligible minutes, greedy maximum number of alerts with spacing >= 240 (K_c), count nu_c, nu_c / K_c',
                    'odds_ratios': 'exact step odds ratio OR_v(t, H) = odds under weights v / odds under Q at every free decision of the observed M0 path and of N_DRAW exact draws from Q; '
                                   'max |log OR| / log Gamma per weighting; each value is a lower bound on the design constant kappa_D(Gamma)',
                    'weighted_tails': 'exact P_{Q_v}(T >= T_obs) under fixed weightings v on a Gamma grid; each is a lower bound on the Rosenbaum-type worst-case p, hence an upper bound on Gamma*'},
        'weightings': {'tilt': 'Gamma on pre-onset window minutes (h = 1)',
                       'phase': 'Gamma on minutes whose 240-minute phase since the month start is even',
                       'tilt_avoid': 'Gamma on pre-onset minutes, 1 on the 239 minutes before each pre-onset window start, sqrt(Gamma) elsewhere',
                       'avoid_block': '1 on the 239 minutes before each pre-onset window start, Gamma elsewhere',
                       'random_blocks': f'Gamma on random {BLOCK_RAND_MIN}-minute blocks (probability 1/2, seed {SEED})',
                       'cell_constant': 'check only: a constant per (month, decile) cell; every admissible set has the same weight, so OR must equal 1'},
        'gamma_or': list(GAMMA_OR), 'gamma_tail': list(GAMMA_TAIL), 'n_draw': N_DRAW, 'lambda_star_nb35': LAMBDA_STAR,
        'readings': 'kappa lower bounds and Gamma* upper bounds only; no lower bound on Gamma* follows from this notebook (that needs a proof of kappa_D(Gamma) <= Gamma^k); '
                    'packing is descriptive; development sample was inspected repeatedly, so every output is exploratory',
        'design_export': '39_design.npz: minute stamps, month index, decile, pre-onset flag, counts per cell, M0 alert positions; no prices',
        'seed': SEED, 'notebook16_spec_sha256': FROZEN16['spec_sha256'], 'holdouts': 'not used', 'prospective_folder': 'not read',
        'status': 'exploratory; appendix; no confirmatory plan changed'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True, default=str) + '\n').encode()
_sp = OUT / '39_SPEC.json'
if _sp.exists() and _sp.read_bytes() != _raw:
    if (OUT / '5_weighted_tail.csv').exists():
        raise RuntimeError('SPEC changed but results from the replaced SPEC exist; move them out of the folder first')
    _sp.rename(OUT / f'39_SPEC.replaced_{pd.Timestamp.now(tz="UTC"):%Y%m%dT%H%M%S}.json')
_sp.write_bytes(_raw); SPEC_SHA = hashlib.sha256(_raw).hexdigest()
print('Output:', OUT, '| spec sha256', SPEC_SHA[:16])

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/design_constant_v1 | spec sha256 7d2d33b4953e712e


## 0. Reproduce M0 scores, alerts and events

I use the same code as section 0 of notebooks 34 and 35.

In [2]:
t_ = time.time()
px, sg = load_btc_spot()
F = causal_features(px, sg)
T = px.index.asi8; close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
v_ = sg.z_vol.to_numpy(float); f_ = sg.tsr.to_numpy(float); h_ = sg.z_hs.to_numpy(float)
VALID = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v_) & np.isfinite(f_) & np.isfinite(h_) & (f_ >= 0) & (f_ <= 1))
EVP = VALID & in_period(T, EVAL_START, EVAL_END)
_lc = np.log(np.where(np.isfinite(close) & (close > 0), close, np.nan))
RV = pd.Series(_lc).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
TCOLS = ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']
E_ref = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
CAT = price_only_catalog(close, volume, px.index, H=H_REF, decline=DCL)
for c_ in TCOLS:
    CAT[c_] = pd.to_datetime(CAT[c_], utc=True)
CAT = CAT.sort_values('breach_utc').reset_index(drop=True)
if len(CAT) != len(E_ref) or any(not np.array_equal(CAT[c_].astype('int64').to_numpy(), E_ref[c_].astype('int64').to_numpy()) for c_ in TCOLS):
    raise RuntimeError('GUARD FAIL: H=240 catalog differs from the notebook-2 catalog')
EV = events_between(CAT, EVAL_START, EVAL_END).reset_index(drop=True)
Xl = F[['z_vol', 'tsr']].to_numpy(float); okl = train_mask(T, VALID) & np.isfinite(Xl).all(axis=1)
cfg0 = FROZEN16['configs'][f'M0_VF_logit|{DNAME}']
m0 = new_logit().fit(Xl[okl], label_from_catalog(T, CAT)[okl])
if not (np.allclose(m0[-1].coef_.ravel(), cfg0['coef'], rtol=1e-6, atol=1e-9) and np.isclose(m0[-1].intercept_[0], cfg0['intercept'], rtol=1e-6, atol=1e-9)):
    raise RuntimeError('GUARD FAIL: M0 refit differs from notebook 16')
TH0 = float(cfg0['threshold_2022'])
S0_FULL = np.full(len(T), np.nan); okf = EVP & np.isfinite(Xl).all(axis=1); S0_FULL[okf] = m0.predict_proba(Xl[okf])[:, 1]
A_M0 = np.sort(pd.to_datetime(pd.read_csv(M16 / f'alarms_M0_VF_logit_{DNAME}.csv').alarm_utc, utc=True).astype('int64').to_numpy())
if len(A_M0) != cfg0['dev_alerts']:
    raise RuntimeError('GUARD FAIL: notebook-16 M0 alarms changed')
_ok0 = EVP & np.isfinite(S0_FULL)
if not np.array_equal(emit_calendar_fast(T[_ok0], S0_FULL[_ok0] >= TH0), A_M0):
    raise RuntimeError('GUARD FAIL: re-running the frozen scheduler on the M0 score does not reproduce the notebook-16 alerts')
del Xl
print(f'PASS: M0 model, alerts and scheduler verified; {len(EV)} events in the development period ({time.time() - t_:.0f}s)')


def ev_arrays(ev):
    return dict(br=ev.breach_utc.astype('int64').to_numpy(), on=ev.onset_1pct_utc.astype('int64').to_numpy(), pk=ev.peak_utc.astype('int64').to_numpy(), n=len(ev))


def fast_counts(a, ea):
    a = np.sort(np.asarray(a, np.int64)); used = np.zeros(len(a), bool); pre = post = 0
    for k in range(ea['n']):
        b = ea['br'][k]; l_ = np.searchsorted(a, b - LEAD_MAX * NS, 'left'); r_ = np.searchsorted(a, b - LEAD_MIN * NS, 'right')
        for j in range(l_, r_):
            if not used[j]:
                used[j] = True
                if a[j] < ea['on'][k]:
                    pre += 1
                else:
                    post += 1
                break
    return dict(pre=pre, post=post)


def make_space(Tp, RVp, start, end):
    okd = (Tp >= start.value) & (Tp < end.value) & np.isfinite(RVp)
    tt = Tp[okd]; dq = np.minimum((pd.Series(RVp[okd]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    mo = cal_fields(tt)[1]; key = mo * 10 + dq; o2 = np.lexsort((tt, key))
    return dict(TT=tt, DD=dq, PK=tt[o2], KS=key[o2])


EA4 = ev_arrays(EV); SP = make_space(T, np.where(VALID, RV, np.nan), ts(EVAL_START), ts(EVAL_END))
mm0 = match_events(EV, A_M0); fc0 = fast_counts(A_M0, EA4)
if (int(mm0.pre1pct.sum()), int((~mm0.pre1pct).sum())) != (fc0['pre'], fc0['post']):
    raise RuntimeError('GUARD FAIL: fast matcher differs from match_events')
if fc0['pre'] != int(REF34.pre_observed):
    raise RuntimeError('GUARD FAIL: M0 pre-onset catches differ from notebook 34')
print('M0 pre-onset catches:', fc0['pre'], '| grid minutes:', len(SP['TT']))

PASS: M0 model, alerts and scheduler verified; 83 events in the development period (25s)
M0 pre-onset catches: 22 | grid minutes: 1750857


## 1. Kernel and check

I run the backward recursion of notebook 34's exact null (exactnull) on K weight columns at once. At every free decision point on a path (outside the cooldown, where both choices are possible), the recursion turns the weighted sums for placing and skipping into OR_v = (take_v / skip_v) / (take_1 / skip_1). On small random designs I also compute the same values directly by enumerating every placement and compare the two.

In [3]:
"""Exact conditional null for budget-constrained alerts (notebook 34 core).
Q_nu: uniform over alert sets A ⊂ positions with spacing >= C (time units) and exact counts nu[month, class].
Positions are sorted times; each has a month index, a class, an indicator h (pre-onset minute) and a weight w (1 = uniform).
Statistic T(A) = sum_{t in A} h(t) (additive because the judging window is shorter than the cooldown; checked separately on the real data)."""
import numpy as np
from numba import njit

# ---------------------------------------------------------------- design preparation
def prepare(times, month, cls, h, nu, C, w=None):
    """Drop positions whose (month, class) has zero count (they can never hold an alert), build radix states per month.
    month: int array (any labels, increasing in time); cls: int in [0, K); nu: dict {(month_label, class): count}."""
    times = np.asarray(times, np.int64); month = np.asarray(month, np.int64); cls = np.asarray(cls, np.int64); h = np.asarray(h, np.int64)
    assert np.all(np.diff(times) > 0)
    w = np.ones(len(times)) if w is None else np.asarray(w, float)
    K = int(max(cls.max() + 1, max(b for _, b in nu) + 1)) if nu else int(cls.max() + 1)
    mlabels = sorted({m for (m, b), v in nu.items() if v > 0})
    mid = {m: k for k, m in enumerate(mlabels)}; M = len(mlabels)
    NU = np.zeros((M, K), np.int64)
    for (m, b), v in nu.items():
        if v > 0: NU[mid[m], b] = v
    ml = np.array(mlabels, np.int64); pos = np.searchsorted(ml, month); pos_c = np.clip(pos, 0, max(M - 1, 0))
    keep = (M > 0) & (pos < M) & (ml[pos_c] == month) if M else np.zeros(len(month), bool)
    if M: keep &= NU[pos_c, np.clip(cls, 0, K - 1)] > 0
    t = times[keep]; mo = pos_c[keep].astype(np.int64); c = cls[keep]; hh = h[keep]; ww = w[keep]
    if M and not np.all(np.diff(mo) >= 0): raise ValueError('months must be increasing in time')
    L = len(t)
    mstart = np.searchsorted(mo, np.arange(M), 'left'); mend = np.searchsorted(mo, np.arange(M), 'right')
    if np.any(mend <= mstart): raise ValueError('a month with a positive count has no admissible position')
    RAD = NU + 1; STR = np.zeros((M, K), np.int64); S = np.ones(M, np.int64); FULL = np.zeros(M, np.int64)
    for m in range(M):
        s = 1
        for b in range(K):
            if NU[m, b] > 0: STR[m, b] = s; s *= RAD[m, b]
        S[m] = s; FULL[m] = int((NU[m] * STR[m]).sum())
    st = STR[mo, c]; rad = RAD[mo, c]
    nx = np.searchsorted(t, t + C, 'left').astype(np.int64)
    R = int((nx - np.arange(L)).max()) + 1 if L else 1
    return dict(t=t, mon=mo, cls=c, h=hh, w=ww, st=st, rad=rad, nx=nx, S=S, FULL=FULL, mstart=mstart, mend=mend, M=M, L=L, R=R,
                NU=NU, keep=keep, mlabels=mlabels, n=int(NU.sum()))

# ---------------------------------------------------------------- backward pass
@njit(cache=False)
def _pairs(D, mode, hi):
    """(destination bin, source bin) pairs for multiplying by u^hi (mode 0, saturating) or (1 + x)^hi (mode 1, truncated)."""
    if hi == 0:
        P = np.empty((D, 2), np.int64)
        for k in range(D): P[k, 0] = k; P[k, 1] = k
        return P
    if mode == 0:
        P = np.empty((D, 2), np.int64)
        for k in range(D - 1): P[k, 0] = k + 1; P[k, 1] = k
        P[D - 1, 0] = D - 1; P[D - 1, 1] = D - 1
        return P
    P = np.empty((2 * D - 1, 2), np.int64)
    for k in range(D): P[k, 0] = k; P[k, 1] = k
    for k in range(D - 1): P[D + k, 0] = k + 1; P[D + k, 1] = k
    return P

@njit(cache=False)
def _backward(nx, mon, st, rad, h, w, S, FULL, D, mode, R):
    """Backward recursion F(i, r; u) over kept positions. mode 0: saturating counter of T (bins 0..D-2 exact, bin D-1 = '>= D-1');
    mode 1: truncated polynomial in x = u - 1 (coefficients Z, Z E[T], Z E[T(T-1)]/2, ...). Ring values share one running log scale e;
    Gfull[i] is the value at the full state of month(i) with its scale lcfull[i] (true value = Gfull * exp(lcfull))."""
    L = len(nx); M = len(S); Smax = 1
    for m in range(M):
        if S[m] > Smax: Smax = S[m]
    ring = np.zeros((R, D, Smax)); Gfull = np.zeros((L, D)); lcfull = np.full(L, -np.inf)
    P0 = _pairs(D, mode, 0); P1 = _pairs(D, mode, 1); e = 0.0; BIG = 1e200; LBIG = np.log(BIG)
    for i in range(L - 1, -1, -1):
        m = mon[i]; Sm = S[m]; cur = ring[i % R]
        # skip term
        if i + 1 < L and mon[i + 1] == m:
            src = ring[(i + 1) % R]
            for k in range(D):
                for s in range(Sm): cur[k, s] = src[k, s]
        else:
            for k in range(D):
                for s in range(Sm): cur[k, s] = 0.0
            if i + 1 < L:
                if lcfull[i + 1] > -np.inf:
                    f = np.exp(lcfull[i + 1] - e)
                    for k in range(D): cur[k, 0] = f * Gfull[i + 1, k]
            else:
                cur[0, 0] = np.exp(-e)
        # take term
        j = nx[i]; wi = w[i]; stb = st[i]; rb = rad[i]
        PP = P1 if h[i] == 1 else P0
        if j < L and mon[j] == m:
            src = ring[j % R]; block = stb * rb; nq = Sm // block
            for p in range(PP.shape[0]):
                kd = PP[p, 0]; ks = PP[p, 1]
                for q in range(nq):
                    base = q * block
                    for r in range(1, rb):
                        o = base + r * stb
                        for l in range(stb):
                            cur[kd, o + l] += wi * src[ks, o - stb + l]
        elif stb < Sm:
            ok = False; f = 0.0
            if j < L and mon[j] == m + 1 and lcfull[j] > -np.inf:
                ok = True; f = wi * np.exp(lcfull[j] - e)
            elif j >= L and m == M - 1:
                ok = True; f = wi * np.exp(-e)
            if ok:
                for p in range(PP.shape[0]):
                    kd = PP[p, 0]; ks = PP[p, 1]
                    if j < L: cur[kd, stb] += f * Gfull[j, ks]
                    elif ks == 0: cur[kd, stb] += f
        # occasional rescale of the whole ring (values only grow as i decreases)
        if (i & 15) == 0 or (i + 1 < L and mon[i + 1] != m):
            mx = 0.0
            for k in range(D):
                for s in range(Sm):
                    if cur[k, s] > mx: mx = cur[k, s]
            if mx > BIG:
                ring *= 1.0 / BIG; e += LBIG
        f_ = FULL[m]
        for k in range(D): Gfull[i, k] = cur[k, f_]
        lcfull[i] = e
    return Gfull, lcfull

def exact_law(P, kcap):
    """Exact law of T under Q_nu: P(T = k) for k < kcap and P(T >= kcap) (last entry)."""
    G, lc = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], int(kcap) + 1, 0, P['R'])
    v = G[0]
    if not v.sum() > 0: raise ValueError('no admissible configuration: the counts cannot be placed under the cooldown')
    return v / v.sum(), lc[0] + np.log(v.sum())

def exact_moments(P):
    """Exact E[T] and Var[T] under Q_nu from the factorial-moment pass."""
    G, lc = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], 3, 1, P['R'])
    z, f1, f2 = G[0]; mean = f1 / z; e2 = 2 * f2 / z           # E[T(T-1)] = 2 * coefficient of x^2
    return mean, e2 + mean - mean ** 2

def log_count(P):
    """log of the number (weighted) of admissible configurations."""
    G, lc = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], 1, 0, P['R'])
    return lc[0] + np.log(G[0, 0])

# ---------------------------------------------------------------- exact sampler
@njit(cache=False)
def _rows(m, lo, hi, after, after_e, nx, mon, st, rad, w, S, mend, Gfull, lcfull):
    """Backward rows (D = 1, u = 1) of month m for positions [lo, hi), given the rows of [hi, hi + len(after)) (empty when hi is the month end).
    Returns rows for [lo, hi + len(after)); row r holds F(lo + r, .) * exp(-erow[r])."""
    b = mend[m]; Sm = S[m]; L = len(nx); M = len(S); nA = after.shape[0]; tot = hi - lo + nA
    T = np.zeros((tot, Sm)); erow = np.zeros(tot); BIG = 1e200; LBIG = np.log(BIG)
    for r in range(nA):
        erow[hi - lo + r] = after_e[r]
        for s in range(Sm): T[hi - lo + r, s] = after[r, s]
    if nA > 0: e = after_e[0]
    else:
        e = lcfull[b] if b < L else 0.0
        if e == -np.inf: e = 0.0
    for i in range(hi - 1, lo - 1, -1):
        il = i - lo
        if i + 1 < b:
            f = np.exp(erow[il + 1] - e)
            for s in range(Sm): T[il, s] = f * T[il + 1, s]
        elif i + 1 < L:
            if lcfull[i + 1] > -np.inf: T[il, 0] = np.exp(lcfull[i + 1] - e) * Gfull[i + 1, 0]
        else:
            T[il, 0] = np.exp(-e)
        j = nx[i]; stb = st[i]; rb = rad[i]; wi = w[i]
        if j < b:
            f = wi * np.exp(erow[j - lo] - e); block = stb * rb; nq = Sm // block; jl = j - lo
            for q in range(nq):
                base = q * block
                for r in range(1, rb):
                    o = base + r * stb
                    for l in range(stb):
                        T[il, o + l] += f * T[jl, o - stb + l]
        elif stb < Sm:
            if j < L and mon[j] == m + 1 and lcfull[j] > -np.inf: T[il, stb] += wi * np.exp(lcfull[j] - e) * Gfull[j, 0]
            elif j >= L and m == M - 1: T[il, stb] += wi * np.exp(-e)
        mx = 0.0
        for s in range(Sm):
            if T[il, s] > mx: mx = T[il, s]
        if mx > BIG:
            for s in range(Sm): T[il, s] /= BIG
            e += LBIG
        erow[il] = e
    return T, erow

@njit(cache=False)
def _walk_seg(m, lo, hi, T, erow, nx, mon, st, rad, w, mend, FULL, Gfull, lcfull, cur_i, cur_s, out, nout, seed):
    """Advance every draw whose current position lies in [lo, hi) of month m until it leaves the segment."""
    np.random.seed(seed)
    b = mend[m]; L = len(nx); M = len(FULL)
    for d in range(len(cur_i)):
        i = cur_i[d]; s = cur_s[d]
        if i < lo or i >= hi: continue
        while i < hi:
            if s == 0:
                i = b; break
            il = i - lo; j = nx[i]; tv = 0.0; stb = st[i]
            if (s // stb) % rad[i] >= 1:
                if j < b:
                    tv = w[i] * T[j - lo, s - stb] * np.exp(erow[j - lo] - erow[il])
                elif s - stb == 0:
                    if j < L and mon[j] == m + 1: tv = w[i] * Gfull[j, 0] * np.exp(lcfull[j] - erow[il])
                    elif j >= L and m == M - 1: tv = w[i] * np.exp(-erow[il])
            sv = T[il + 1, s] * np.exp(erow[il + 1] - erow[il]) if i + 1 < b else 0.0
            p = tv / (tv + sv) if tv + sv > 0.0 else 0.0
            if np.random.random() < p:
                out[d, nout[d]] = i; nout[d] += 1; s -= stb; i = j
                if j >= b: break
            else:
                i += 1
        cur_i[d] = i
        if i >= b: cur_s[d] = FULL[mon[i]] if i < L else 0
        else: cur_s[d] = s

def sample_exact(P, ndraw, seed=0, seg=4000):
    """ndraw independent exact draws from Q_nu; returns an (ndraw, n) array of kept-position indices (sorted per draw).
    Each month is processed in segments of `seg` positions: one backward pass stores only the rows just after each segment boundary,
    and each segment is recomputed from its checkpoint when the draws walk through it (memory about (seg + R) x states per month)."""
    G, lcf = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], 1, 0, P['R'])
    n = P['n']; R = P['R']; out = np.full((ndraw, n), -1, np.int64); nout = np.zeros(ndraw, np.int64)
    cur_i = np.zeros(ndraw, np.int64); cur_s = np.full(ndraw, P['FULL'][0], np.int64)
    args = (P['nx'], P['mon'], P['st'], P['rad'], P['w'], P['S'], P['mend'], G, lcf)
    for m in range(P['M']):
        a, b = int(P['mstart'][m]), int(P['mend'][m]); Sm = int(P['S'][m])
        bounds = list(range(a, b, seg)) + [b]; nseg = len(bounds) - 1
        ck = [None] * nseg; after = np.zeros((0, Sm)); after_e = np.zeros(0)
        for k in range(nseg - 1, -1, -1):
            lo, hi = bounds[k], bounds[k + 1]
            ck[k] = (after, after_e)
            Tm, er = _rows(m, lo, hi, after, after_e, *args)
            nA = min(R, b - lo); after = Tm[:nA].copy(); after_e = er[:nA].copy()
        for k in range(nseg):
            lo, hi = bounds[k], bounds[k + 1]
            Tm, er = _rows(m, lo, hi, ck[k][0], ck[k][1], *args)
            _walk_seg(m, lo, hi, Tm, er, P['nx'], P['mon'], P['st'], P['rad'], P['w'], P['mend'], P['FULL'], G, lcf,
                      cur_i, cur_s, out, nout, (seed * 1000003 + m) * 1009 + k)
    if np.any(nout != n): raise RuntimeError('sampler did not place every alert')
    return out

def check_draws(P, idx, C):
    """Constraint audit of exact draws: counts per (month, class) and spacing."""
    t = P['t'][idx]; bad_space = int((np.diff(t, axis=1) < C).sum())
    bad_count = 0
    for d in range(idx.shape[0]):
        cnt = np.zeros_like(P['NU']); np.add.at(cnt, (P['mon'][idx[d]], P['cls'][idx[d]]), 1)
        bad_count += int(np.any(cnt != P['NU']))
    return bad_space, bad_count
import itertools, random


@njit(cache=False)
def path_states(idx, nx, mon, FULL, st, L):
    """State (remaining counts of the month, radix code) of the configuration idx at every free decision point; -1 elsewhere."""
    PS = np.full(L, -1, np.int64); k = 0; n = len(idx); free = 0
    if L == 0:
        return PS
    cur = mon[0]; s = FULL[cur]
    for i in range(L):
        if mon[i] != cur:
            cur = mon[i]; s = FULL[cur]
        if i < free:
            continue
        PS[i] = s
        if k < n and idx[k] == i:
            s -= st[i]; free = nx[i]; k += 1
    return PS


@njit(cache=False)
def _or_paths(nx, mon, st, rad, W, S, FULL, R, PS, store):
    """Backward recursion of exactnull._backward (D = 1) for K weight columns at once (column 0 must be all ones).
    At every free decision point of every path (PS[p, i] >= 0) it reads take and skip values and records
    log OR_k = log(take_k / skip_k) - log(take_0 / skip_0). Decisions where the null forces the choice are counted, not used."""
    L = len(nx); M = len(S); K = W.shape[1]; NP = PS.shape[0]; Smax = 1
    for m in range(M):
        if S[m] > Smax: Smax = S[m]
    ring = np.zeros((R, K, Smax)); Gfull = np.zeros((L, K)); lcfull = np.full((L, K), -np.inf)
    e = np.zeros(K); BIG = 1e200; LBIG = np.log(BIG)
    mx = np.full((NP, K), -np.inf); mn = np.full((NP, K), np.inf)
    imx = np.full((NP, K), -1, np.int64); imn = np.full((NP, K), -1, np.int64)
    ndec = np.zeros(NP, np.int64); nforced = np.zeros(NP, np.int64)
    LOR = np.full((NP, L if store else 1, K), np.nan)
    tk = np.zeros(K); sk = np.zeros(K)
    for i in range(L - 1, -1, -1):
        m = mon[i]; Sm = S[m]; cur = ring[i % R]
        j = nx[i]; stb = st[i]; rb = rad[i]
        same_next = (i + 1 < L) and (mon[i + 1] == m)
        take_in = (j < L) and (mon[j] == m)
        for p in range(NP):
            s = PS[p, i]
            if s < 0:
                continue
            for k in range(K):
                if same_next:
                    sv = ring[(i + 1) % R, k, s]
                elif s == 0:
                    if i + 1 < L:
                        sv = np.exp(lcfull[i + 1, k] - e[k]) * Gfull[i + 1, k]
                    else:
                        sv = np.exp(-e[k])
                else:
                    sv = 0.0
                tv = 0.0
                if (s // stb) % rb >= 1:
                    if take_in:
                        tv = W[i, k] * ring[j % R, k, s - stb]
                    elif s == stb:
                        if j < L and mon[j] == m + 1:
                            tv = W[i, k] * np.exp(lcfull[j, k] - e[k]) * Gfull[j, k]
                        elif j >= L and m == M - 1:
                            tv = W[i, k] * np.exp(-e[k])
                tk[k] = tv; sk[k] = sv
            if tk[0] > 0.0 and sk[0] > 0.0:
                ndec[p] += 1
                base = np.log(tk[0]) - np.log(sk[0])
                for k in range(K):
                    lo = np.log(tk[k]) - np.log(sk[k]) - base
                    if store:
                        LOR[p, i, k] = lo
                    if lo > mx[p, k]:
                        mx[p, k] = lo; imx[p, k] = i
                    if lo < mn[p, k]:
                        mn[p, k] = lo; imn[p, k] = i
            else:
                nforced[p] += 1
        for k in range(K):
            if same_next:
                for s in range(Sm): cur[k, s] = ring[(i + 1) % R, k, s]
            else:
                for s in range(Sm): cur[k, s] = 0.0
                if i + 1 < L:
                    cur[k, 0] = np.exp(lcfull[i + 1, k] - e[k]) * Gfull[i + 1, k]
                else:
                    cur[k, 0] = np.exp(-e[k])
            wi = W[i, k]
            if take_in:
                block = stb * rb; nq = Sm // block; jr = j % R
                for q in range(nq):
                    bq = q * block
                    for r in range(1, rb):
                        o = bq + r * stb
                        for l in range(stb):
                            cur[k, o + l] += wi * ring[jr, k, o - stb + l]
            elif stb < Sm:
                if j < L and mon[j] == m + 1:
                    cur[k, stb] += wi * np.exp(lcfull[j, k] - e[k]) * Gfull[j, k]
                elif j >= L and m == M - 1:
                    cur[k, stb] += wi * np.exp(-e[k])
            if (i & 15) == 0 or not same_next:
                mxv = 0.0
                for s in range(Sm):
                    if cur[k, s] > mxv: mxv = cur[k, s]
                if mxv > BIG:
                    for rr in range(R):
                        for s in range(Smax): ring[rr, k, s] /= BIG
                    e[k] += LBIG
            Gfull[i, k] = cur[k, FULL[m]]; lcfull[i, k] = e[k]
    return mx, mn, imx, imn, ndec, nforced, Gfull, lcfull, LOR


def or_paths(P, W, paths, store=False):
    """paths: list of sorted kept-position index arrays (each an admissible configuration)."""
    PS = np.stack([path_states(np.asarray(a, np.int64), P['nx'], P['mon'], P['FULL'], P['st'], P['L']) for a in paths])
    return _or_paths(P['nx'], P['mon'], P['st'], P['rad'], np.ascontiguousarray(W, dtype=np.float64), P['S'], P['FULL'], P['R'], PS, store)


def _enumerate(times, month, cls, nu, C):
    n = sum(nu.values()); out = []
    for comb in itertools.combinations(range(len(times)), n):
        if any(times[comb[k + 1]] - times[comb[k]] < C for k in range(n - 1)):
            continue
        cnt = {}
        for i in comb:
            cnt[(month[i], cls[i])] = cnt.get((month[i], cls[i]), 0) + 1
        if all(cnt.get(c, 0) == v for c, v in nu.items() if v > 0) and all(nu.get(c, 0) == v for c, v in cnt.items()):
            out.append(comb)
    return out


def _brute_or(A, t, Wcol, path, C):
    """Exact log OR along one admissible configuration `path` (kept indices) by enumeration; NaN where not a free, unforced decision."""
    sets = [tuple(a) for a in A]; L_ = len(t); out = np.full(L_, np.nan); pset = set(path); last = None
    wt = lambda a, w: float(np.prod([w[x] for x in a]))
    for i in range(L_):
        if last is not None and t[i] - t[last] < C:
            if i in pset: last = i
            continue
        H = tuple(x for x in path if x < i)
        comp = [a for a in sets if tuple(x for x in a if x < i) == H]
        z = {}
        for key, w in (('1', np.ones(L_)), ('v', Wcol)):
            z[key + 't'] = sum(wt(a, w) for a in comp if i in a); z[key + 's'] = sum(wt(a, w) for a in comp if i not in a)
        if z['1t'] > 0 and z['1s'] > 0:
            out[i] = np.log(z['vt'] / z['vs']) - np.log(z['1t'] / z['1s'])
        if i in pset: last = i
    return out


random.seed(SEED); _r = np.random.default_rng(SEED)
krows = []; t_ = time.time()
for trial in range(400):
    L_ = random.randint(5, 12); times = np.cumsum([random.choice([1, 1, 1, 2, 3]) for _ in range(L_)])
    M_ = random.randint(1, 3); K_ = random.randint(1, 3); C_ = random.randint(1, 5)
    month = np.sort(_r.integers(0, M_, L_)) * 7 + 3; cls = _r.integers(0, K_, L_); h = np.zeros(L_, int)
    nu = {c: random.randint(0, 2) for c in sorted(set(zip(month.tolist(), cls.tolist())))}
    if sum(nu.values()) == 0:
        continue
    try:
        P_ = prepare(times, month, cls, h, nu, C_)
    except ValueError:
        continue
    kt = P_['t']; nu_idx = {(m_, b_): int(P_['NU'][m_, b_]) for m_ in range(P_['M']) for b_ in range(P_['NU'].shape[1]) if P_['NU'][m_, b_] > 0}
    A_ = _enumerate(kt.tolist(), P_['mon'].tolist(), P_['cls'].tolist(), nu_idx, C_)
    if not A_:
        continue
    G_ = random.choice([1.5, 2.0, 3.0])
    Wv = np.column_stack([np.ones(P_['L']), np.where(_r.random(P_['L']) < 0.5, G_, 1.0), 1 + (G_ - 1) * _r.random(P_['L'])])
    paths = [A_[random.randrange(len(A_))] for _ in range(2)]
    mxk, mnk, _a, _b, nd, nf, Gk, lck, LORk = or_paths(P_, Wv, paths, store=True)
    err = 0.0
    for p_, path in enumerate(paths):
        for k in (1, 2):
            ref = _brute_or(A_, kt, Wv[:, k], list(path), C_); got = LORk[p_, :, k]
            if not np.array_equal(np.isnan(ref), np.isnan(got)):
                raise RuntimeError(f'Decision points differ from enumeration (trial {trial})')
            if np.any(~np.isnan(ref)):
                err = max(err, float(np.nanmax(np.abs(ref - got))))
            _z = LORk[p_, :, 0]
            if np.any(~np.isnan(_z)) and np.nanmax(np.abs(_z)) > 1e-12:
                raise RuntimeError('The all-ones column does not give OR = 1')
    lc_ref = log_count(P_); lc_k = lck[0, 0] + np.log(Gk[0, 0])
    krows.append(dict(trial=trial, configs=len(A_), cooldown=C_, err=err, logcount_err=abs(lc_ref - lc_k), decisions=int(nd.sum()), forced=int(nf.sum())))
KC = pd.DataFrame(krows); KC.to_csv(OUT / '1_kernel_check.csv', index=False)
print(f'{len(KC)} small designs: max |log OR| error {KC.err.max():.1e}; log-count error {KC.logcount_err.max():.1e}; '
      f'{KC.decisions.sum()} decisions checked ({time.time() - t_:.0f}s)')
if not (KC.err.max() < 1e-9 and KC.logcount_err.max() < 1e-9 and KC.decisions.sum() > 500):
    raise RuntimeError('The odds-ratio kernel failed the small-problem verification')
print('PASS: odds-ratio kernel agrees with full enumeration')

208 small designs: max |log OR| error 1.1e-15; log-count error 0.0e+00; 1318 decisions checked (9s)
PASS: odds-ratio kernel agrees with full enumeration


## 2. Actual design, reproduction check and design export

I build the grid, cells and pre-onset minutes with the same code as notebooks 34 and 35 and then export the design.

In [4]:
TT = SP['TT']; GRID_MIN = (TT // NS).astype(np.int64); GRID_MON = cal_fields(TT)[1]; GRID_DEC = SP['DD'].astype(np.int64)
lo_w = EA4['br'] - LEAD_MAX * NS; hi_w = EA4['br'] - LEAD_MIN * NS
if np.any(lo_w[1:] <= hi_w[:-1]) or (LEAD_MAX - LEAD_MIN) >= COOLDOWN:
    raise RuntimeError('GUARD FAIL: T is not additive')
H_GRID = np.zeros(len(TT), np.int64)
for k in range(EA4['n']):
    l_ = np.searchsorted(TT, lo_w[k], 'left'); r_ = np.searchsorted(TT, hi_w[k], 'right')
    H_GRID[l_:r_] = (TT[l_:r_] < EA4['on'][k]).astype(np.int64)


def cells_of(a):
    a = np.asarray(a, np.int64); j = np.clip(np.searchsorted(TT, a, 'right') - 1, 0, len(TT) - 1)
    return cal_fields(a)[1], GRID_DEC[j], TT[j] == a


def design_for(a):
    mo, dq, on_grid = cells_of(a)
    nu = {}
    for m_, d_ in zip(mo.tolist(), dq.tolist()):
        nu[(m_, d_)] = nu.get((m_, d_), 0) + 1
    return prepare(GRID_MIN, GRID_MON, GRID_DEC, H_GRID, nu, C_MIN), int((~on_grid).sum())


def t_additive(a):
    a = np.asarray(a, np.int64); k = np.searchsorted(lo_w, a, 'right') - 1; ok = k >= 0
    kk = np.clip(k, 0, len(lo_w) - 1)
    return int((ok & (a <= hi_w[kk]) & (a < EA4['on'][kk])).sum())


P0, off0 = design_for(A_M0); T0 = fc0['pre']
if t_additive(A_M0) != T0:
    raise RuntimeError('GUARD FAIL: sum of h over M0 alerts differs from fast_counts pre')
if off0:
    raise RuntimeError(f'GUARD FAIL: {off0} M0 alerts are off the placement grid')
IDX_M0 = np.searchsorted(P0['t'], A_M0 // NS)
if not (np.all(IDX_M0 < P0['L']) and np.array_equal(P0['t'][IDX_M0], A_M0 // NS)) or int(P0['h'][IDX_M0].sum()) != T0:
    raise RuntimeError('GUARD FAIL: M0 alerts do not map onto kept positions with T = T_obs')
_mem_or = (P0['R'] * int(P0['S'].max()) + 2 * P0['L']) * 8 / 2 ** 30
print(f'M0 design: {P0["n"]} alerts, {P0["M"]} months, positions {P0["L"]}, state sizes median {int(np.median(P0["S"]))} / max {int(P0["S"].max())}; '
      f'memory per weight column about {_mem_or:.2f} GiB')
t_ = time.time()
MEAN_EXACT, VAR_EXACT = exact_moments(P0)
rel_mean = abs(MEAN_EXACT - float(REF34.exact_mean)) / float(REF34.exact_mean)
print(f'exact E[T] {MEAN_EXACT:.6f} vs notebook 34 {float(REF34.exact_mean):.6f} (relative difference {rel_mean:.1e}; {time.time() - t_:.0f}s)')
if rel_mean > 1e-9:
    raise RuntimeError('The design does not reproduce the notebook-34 exact mean')
LOGCOUNT = log_count(P0)
np.savez_compressed(OUT / '39_design.npz', t_min=P0['t'], month_index=P0['mon'], decile=P0['cls'], pre_onset=P0['h'].astype(np.int8),
                    counts=P0['NU'], month_labels=np.array(P0['mlabels']), m0_positions=IDX_M0, cooldown_min=C_MIN, T_obs=T0,
                    breach_min=EA4['br'] // NS, onset_min=EA4['on'] // NS, note='kept positions of the notebook-34 placement grid; no prices')
print('saved', OUT / '39_design.npz')

M0 design: 800 alerts, 40 months, positions 1414298, state sizes median 5040 / max 25920; memory per weight column about 0.07 GiB
exact E[T] 6.700735 vs notebook 34 6.700735 (relative difference 0.0e+00; 80s)
saved /content/drive/MyDrive/btc_lead_cost_results_77months_spot/design_constant_v1/39_design.npz


## 3. Cell packing

For each cell c = (month, decile) I record the available minutes N_c, the actual count ν_c and the maximum number of alerts K_c, which I obtain by picking greedily from the start with 240-minute spacing among those minutes alone. Greedy picking is optimal in one dimension; because I ignore spacing to alerts in other cells, K_c is the capacity of the cell alone. I then repeat the same computation per month on the minutes of all its cells combined.

In [5]:
def greedy_capacity(tmin, C):
    k = 0; last = None
    for x in tmin:
        if last is None or x - last >= C:
            k += 1; last = x
    return k


crow = []
for m in range(P0['M']):
    a, b = int(P0['mstart'][m]), int(P0['mend'][m]); tm = P0['t'][a:b]; cm = P0['cls'][a:b]
    for d in np.nonzero(P0['NU'][m])[0]:
        tc = tm[cm == d]; Kc = greedy_capacity(tc, C_MIN); nu_c = int(P0['NU'][m, d])
        crow.append(dict(month=P0['mlabels'][m], decile=int(d), eligible_minutes=len(tc), span_days=float((tc[-1] - tc[0]) / 1440), K_c=Kc, nu_c=nu_c,
                         utilisation=nu_c / Kc, slack=Kc - nu_c))
CP = pd.DataFrame(crow); CP.to_csv(OUT / '3_cell_packing.csv', index=False)
mrow = []
for m in range(P0['M']):
    a, b = int(P0['mstart'][m]), int(P0['mend'][m]); Km = greedy_capacity(P0['t'][a:b], C_MIN); nu_m = int(P0['NU'][m].sum())
    mrow.append(dict(month=P0['mlabels'][m], cells=int((P0['NU'][m] > 0).sum()), alerts=nu_m, K_month=Km, utilisation=nu_m / Km, state_size=int(P0['S'][m])))
MP = pd.DataFrame(mrow); MP.to_csv(OUT / '3b_month_packing.csv', index=False)
print(f'{len(CP)} cells: alerts per cell median {CP.nu_c.median():.0f} (max {CP.nu_c.max()}); utilisation median {CP.utilisation.median():.2f}, '
      f'90th percentile {CP.utilisation.quantile(0.9):.2f}, max {CP.utilisation.max():.2f}; cells at capacity (slack 0): {(CP.slack == 0).sum()}, slack <= 1: {(CP.slack <= 1).sum()}')
print(f'{len(MP)} months: utilisation median {MP.utilisation.median():.2f}, max {MP.utilisation.max():.2f}')
display(CP.sort_values('utilisation', ascending=False).head(10))

262 cells: alerts per cell median 2 (max 17); utilisation median 0.09, 90th percentile 0.21, max 0.83; cells at capacity (slack 0): 0, slack <= 1: 2
40 months: utilisation median 0.13, max 0.21


,month,decile,eligible_minutes,span_days,K_c,nu_c,utilisation,slack
63,24285,6,989,12.112500,6,5,0.833333,1
88,24288,9,1391,0.965278,6,4,0.666667,2
221,24309,5,425,3.303472,3,2,0.666667,1
209,24307,6,1701,4.925694,9,4,0.444444,5
189,24304,9,6372,7.882639,28,12,0.428571,16
36,24281,9,1193,0.827778,5,2,0.400000,3
121,24293,9,2603,1.806944,11,4,0.363636,7
64,24285,7,2078,12.167361,11,4,0.363636,7
137,24295,9,5217,24.227083,23,8,0.347826,15
249,24314,9,12351,25.048611,54,17,0.314815,37


## 4. Step odds ratios under fixed weightings (lower bounds on the design constant κ_D(Γ))

I compute OR_v at every free decision point of the observed M0 path and of N_DRAW paths drawn exactly from Q, with the weight-1 column as reference. The cell-constant weighting gives every admissible placement the same weight, so it serves as a check that OR = 1. Points where Q forces the choice (no alerts left in the cell, or all remaining alerts must be placed) have no defined odds ratio, and I only count them.

In [6]:
Tm = P0['t']; Hm = P0['h'].astype(bool)
_ws = np.array(sorted(set((EA4['br'] // NS - LEAD_MAX).tolist())), np.int64)
_ws = _ws[[bool(Hm[(Tm >= w0) & (Tm <= w0 + (LEAD_MAX - LEAD_MIN))].any()) for w0 in _ws]] if len(_ws) else _ws   # windows with a pre-onset minute
BLOCKZ = np.zeros(P0['L'], bool)
for w0 in _ws:
    l_ = np.searchsorted(Tm, w0 - (C_MIN - 1), 'left'); r_ = np.searchsorted(Tm, w0, 'left'); BLOCKZ[l_:r_] = True
_mstart_min = np.array([Tm[int(P0['mstart'][m])] for m in range(P0['M'])])[P0['mon']]
PHASE = (((Tm - _mstart_min) // C_MIN) % 2 == 0)
_rb = np.random.default_rng(SEED); _blk = (Tm - Tm[0]) // BLOCK_RAND_MIN; RANDB = _rb.random(int(_blk.max()) + 1)[_blk] < 0.5


def weighting(name, g):
    if name == 'tilt':
        return np.where(Hm, g, 1.0)
    if name == 'phase':
        return np.where(PHASE, g, 1.0)
    if name == 'tilt_avoid':
        return np.where(Hm, g, np.where(BLOCKZ, 1.0, np.sqrt(g)))
    if name == 'avoid_block':
        return np.where(BLOCKZ, 1.0, g)
    if name == 'random_blocks':
        return np.where(RANDB, g, 1.0)
    raise KeyError(name)


NAMES_OR = ['tilt', 'phase', 'tilt_avoid', 'avoid_block', 'random_blocks']
COLS = [('ones', 1.0), ('cell_constant', np.nan)] + [(n_, g_) for g_ in GAMMA_OR for n_ in NAMES_OR]
_cc = np.random.default_rng(SEED + 1).uniform(1.0, 2.0, size=P0['NU'].shape)[P0['mon'], P0['cls']]
WALL = np.column_stack([np.ones(P0['L']), _cc] + [weighting(n_, g_) for n_, g_ in COLS[2:]])
t_ = time.time(); DRAWS = sample_exact(P0, N_DRAW, seed=SEED)
bad_s, bad_c = check_draws(P0, DRAWS, C_MIN)
if bad_s or bad_c:
    raise RuntimeError('Exact draws violate the constraints')
PATHS = [IDX_M0] + [DRAWS[d] for d in range(N_DRAW)]
print(f'{N_DRAW} exact draws from Q ({time.time() - t_:.0f}s); weighting columns {len(COLS)}')
try:
    import psutil; _avail = psutil.virtual_memory().available / 2 ** 30
except ImportError:
    _mi = dict(l.split(':', 1) for l in open('/proc/meminfo')); _avail = float(_mi['MemAvailable'].split()[0]) / 2 ** 20
_per_col = _mem_or + len(PATHS) * P0['L'] * 8 / 2 ** 30 / 4
KB = max(2, min(len(COLS), int(0.5 * _avail / max(_per_col, 1e-9))))
print(f'available memory {_avail:.1f} GiB; columns per pass {KB}')
ROWS = []; t_ = time.time()
_rest = list(range(1, len(COLS)))
while _rest:
    batch = [0] + _rest[:KB - 1]; _rest = _rest[KB - 1:]
    mxb, mnb, imx, imn, nd, nf, Gb, lcb, _ = or_paths(P0, WALL[:, batch], PATHS)
    if abs(lcb[0, 0] + np.log(Gb[0, 0]) - LOGCOUNT) > 1e-6 * max(1.0, abs(LOGCOUNT)):
        raise RuntimeError('GUARD FAIL: the odds-ratio kernel count differs from the notebook-34 kernel')
    for jb, c in enumerate(batch):
        name, g = COLS[c]
        for p_ in range(len(PATHS)):
            ROWS.append(dict(weighting=name, gamma=g, path='observed M0' if p_ == 0 else f'draw {p_}', decisions=int(nd[p_]), forced=int(nf[p_]),
                             max_log_or=float(mxb[p_, jb]), min_log_or=float(mnb[p_, jb]),
                             t_max=int(Tm[imx[p_, jb]]) if imx[p_, jb] >= 0 else -1, t_min=int(Tm[imn[p_, jb]]) if imn[p_, jb] >= 0 else -1))
    print(f'  pass with columns {batch}: {time.time() - t_:.0f}s', flush=True)
OR = pd.DataFrame(ROWS)
OR['abs_log_or'] = OR[['max_log_or', 'min_log_or']].abs().max(axis=1)
OR['m_eff'] = np.where(OR.gamma > 1, OR.abs_log_or / np.log(OR.gamma.where(OR.gamma > 1, np.e)), np.nan)
OR.to_csv(OUT / '4_path_odds_ratios.csv', index=False)
_chk = OR[OR.weighting.isin(['ones', 'cell_constant'])].abs_log_or.max()
if _chk > 1e-8:
    raise RuntimeError(f'Ones or cell-constant weights give |log OR| = {_chk:.1e}, not 0')
_kr = []
for g_, d_ in OR[OR.gamma > 1].groupby('gamma'):
    r_ = d_.loc[d_.m_eff.idxmax()]
    _kr.append(dict(gamma=g_, m_eff=float(r_.m_eff), weighting=r_.weighting, path=r_.path, kappa_lower=float(np.exp(d_.abs_log_or.max()))))
KAPPA = pd.DataFrame(_kr).set_index('gamma')
display(OR[OR.gamma > 1].groupby(['gamma', 'weighting']).agg(m_eff_max=('m_eff', 'max'), m_eff_observed=('m_eff', 'first')))
display(KAPPA)
print('free decision points on the observed path where Q leaves a real choice:', int(OR.decisions.iloc[0]), '| where Q forces the step (no count left in the cell, or every remaining count must be placed):', int(OR.forced.iloc[0]))

10 exact draws from Q (238s); weighting columns 12
available memory 9.6 GiB; columns per pass 12
  pass with columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]: 390s


m_eff_max  m_eff_observed
gamma weighting                               
1.5   avoid_block     1.088244        1.047241
      phase           1.493950        1.129615
      random_blocks   1.441660        1.441660
      tilt            1.364174        1.008034
      tilt_avoid      0.977412        0.840742
2.0   avoid_block     1.082077        1.043714
      phase           1.528757        1.130661
      random_blocks   1.504745        1.504745
      tilt            1.334447        1.008782
      tilt_avoid      0.978919        0.848301

,m_eff,weighting,path,kappa_lower
gamma,,,,
1.5,1.493950,phase,draw 9,1.832616
2.0,1.528757,phase,draw 9,2.885371


free decision points on the observed path where Q leaves a real choice: 488390 | where Q forces the step (no count left in the cell, or every remaining count must be placed): 741879


## 5. Exact tail probabilities under fixed weightings (upper bounds on Γ*)

Here I compute P(T ≥ T_obs) under the exact law Q_v of each weighting v. I first check the tilt at Γ = 2 against notebook 35, then compute the other weightings at Γ = 2.5 and 3.0 and save them to `5_weighted_tail.csv`. A bound better than the tilt's Γ* ≤ 3.17 needs p ≥ 0.05 at some Γ below 3.17. Since no two weights in the box [1, Γ] can differ by more than a factor Γ, no weighting can lower the blocked stretches further while keeping the tilt's ratio Γ between pre-onset and other minutes; the two weightings I use here reduce the blocking at that cost.

In [7]:
tpath = OUT / '5_weighted_tail.csv'
TW = pd.read_csv(tpath) if tpath.exists() else pd.DataFrame(columns=['weighting', 'gamma', 'p', 'seconds'])
done_t = {(w_, round(float(g_), 6)): float(p_) for w_, g_, p_ in zip(TW.weighting, TW.gamma, TW.p)}


def p_weighted(name, g):
    key = (name, round(float(g), 6))
    if key in done_t:
        return done_t[key]
    Q = dict(P0); Q['w'] = weighting(name, g); t0_ = time.time()
    law_, _ = exact_law(Q, T0 + 1); p_ = float(law_[T0:].sum()); sec = time.time() - t0_
    done_t[key] = p_
    pd.DataFrame([dict(weighting=name, gamma=key[1], p=p_, seconds=sec)]).to_csv(tpath, mode='a', header=not tpath.exists(), index=False)
    print(f'  {name} Gamma {g}: P(T >= {T0}) = {p_:.4e} ({sec / 60:.1f} min)', flush=True)
    return p_


_p35 = float(TILT35.loc[np.isclose(TILT35.gamma, 2.0), TILT35.columns[1]].iloc[0])
_p2 = p_weighted('tilt', 2.0)
if abs(_p2 - _p35) > 1e-6 * _p35:
    raise RuntimeError(f'GUARD FAIL: tilt at Gamma = 2 gives {_p2:.6e}, notebook 35 gives {_p35:.6e}')
print('PASS: tilt at Gamma = 2 reproduces notebook 35')
for name in ('tilt_avoid', 'avoid_block'):
    for g in GAMMA_TAIL:
        p_weighted(name, g)
TW = pd.read_csv(tpath)
GAMMA_TILT_STAR = float(GSTAR35.loc[np.isclose(GSTAR35.alpha, 0.05), GSTAR35.columns[1]].iloc[0])
UP = []
for name, d in TW.groupby('weighting'):
    over = d[d.p >= 0.05].gamma
    UP.append(dict(weighting=name, smallest_grid_gamma_with_p_over_005=float(over.min()) if len(over) else np.nan))
UP.append(dict(weighting='tilt (notebook 35, bisection)', smallest_grid_gamma_with_p_over_005=GAMMA_TILT_STAR))
UP = pd.DataFrame(UP); UP.to_csv(OUT / '5b_gamma_star_upper.csv', index=False)
display(TW.pivot_table(index='gamma', columns='weighting', values='p')); display(UP)

  tilt Gamma 2.0: P(T >= 22) = 7.8465e-04 (13.7 min)
PASS: tilt at Gamma = 2 reproduces notebook 35
  tilt_avoid Gamma 2.5: P(T >= 22) = 1.1401e-04 (13.8 min)
  tilt_avoid Gamma 3.0: P(T >= 22) = 3.8170e-04 (13.7 min)
  avoid_block Gamma 2.5: P(T >= 22) = 4.5561e-07 (13.8 min)
  avoid_block Gamma 3.0: P(T >= 22) = 5.7486e-07 (13.7 min)


weighting,avoid_block,tilt,tilt_avoid
gamma,,,
2.0,NaN,0.000785,NaN
2.5,4.556079e-07,NaN,0.000114
3.0,5.748626e-07,NaN,0.000382


,weighting,smallest_grid_gamma_with_p_over_005
0,avoid_block,NaN
1,tilt,NaN
2,tilt_avoid,NaN
3,"tilt (notebook 35, bisection)",3.170367


## 6. Summary

In [8]:
GAMMA_UP = float(np.nanmin(UP.smallest_grid_gamma_with_p_over_005))
summary = dict(spec_sha256=SPEC_SHA, run_time=str(pd.Timestamp.now(tz='UTC').floor('s')), status=SPEC['status'],
               checks=dict(kernel_small_designs=int(len(KC)), kernel_max_error=float(KC.err.max()), exact_mean_relative_difference=float(rel_mean),
                           ones_and_cell_constant_max_abs_log_or=float(_chk), tilt_gamma2_vs_nb35=dict(this=_p2, nb35=_p35)),
               design=dict(alerts=int(P0['n']), months=int(P0['M']), positions=int(P0['L']), T_obs=int(T0), max_state_size=int(P0['S'].max())),
               packing=dict(cells=int(len(CP)), alerts_per_cell_median=float(CP.nu_c.median()), alerts_per_cell_max=int(CP.nu_c.max()),
                            utilisation_median=float(CP.utilisation.median()), utilisation_p90=float(CP.utilisation.quantile(0.9)),
                            utilisation_max=float(CP.utilisation.max()), cells_at_capacity=int((CP.slack == 0).sum()), month_utilisation_max=float(MP.utilisation.max())),
               kappa_lower_bounds={str(g): dict(m_eff=float(r.m_eff), kappa_lower=float(r.kappa_lower), weighting=r.weighting, path=r.path) for g, r in KAPPA.iterrows()},
               inclusion_reading={str(g): dict(best_certifiable_gamma_via_inclusion_at_most=float(LAMBDA_STAR ** (1 / max(float(r.m_eff), 1.0)))) for g, r in KAPPA.iterrows()},
               gamma_star_upper=dict(bound=GAMMA_UP, table=UP.to_dict('records')),
               gamma_star_lower='not available from this notebook: needs a proof that kappa_D(Gamma) <= Gamma^k, which would give Gamma* >= 1.795^(1/k)')
(OUT / '39_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o)))
print(json.dumps({k: summary[k] for k in ('packing', 'kappa_lower_bounds', 'inclusion_reading', 'gamma_star_upper')}, indent=1, default=str))
print('Done. Outputs in', OUT)

{
 "packing": {
  "cells": 262,
  "alerts_per_cell_median": 2.0,
  "alerts_per_cell_max": 17,
  "utilisation_median": 0.09090909090909091,
  "utilisation_p90": 0.20620689655172417,
  "utilisation_max": 0.8333333333333334,
  "cells_at_capacity": 0,
  "month_utilisation_max": 0.20618556701030927
 },
 "kappa_lower_bounds": {
  "1.5": {
   "m_eff": 1.4939497677899738,
   "kappa_lower": 1.8326160913178513,
   "weighting": "phase",
   "path": "draw 9"
  },
  "2.0": {
   "m_eff": 1.5287570149750949,
   "kappa_lower": 2.885371365899021,
   "weighting": "phase",
   "path": "draw 9"
  }
 },
 "inclusion_reading": {
  "1.5": {
   "best_certifiable_gamma_via_inclusion_at_most": 1.4795792436253647
  },
  "2.0": {
   "best_certifiable_gamma_via_inclusion_at_most": 1.4664405676564898
  }
 },
 "gamma_star_upper": {
  "bound": 3.170366813732877,
  "table": [
   {
    "weighting": "avoid_block",
    "smallest_grid_gamma_with_p_over_005": NaN
   },
   {
    "weighting": "tilt",
    "smallest_grid_gamma_wi